# GLOBAL VARIABLE

In [1]:
BATCH_SIZE = 4

# Create Custom Dataset for PyTorch

In [2]:
import os
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image

class CustomDataset_general(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.image_folder = os.path.join(root_dir, "images")
        self.mask_folder = os.path.join(root_dir, "masks")
        self.image_files = sorted(os.listdir(self.image_folder))
        self.mask_files = sorted(os.listdir(self.mask_folder))
        self.transform = transform

    def __len__(self):
        return len(self.mask_files)

    def __getitem__(self, idx):
        # Read image
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_folder, img_name)
        image = Image.open(img_path).convert("RGB")
        image_gray = image.convert("L")  # Convert to grayscale

        # Read corresponding mask
        mask_name = self.mask_files[idx]
        mask_path = os.path.join(self.mask_folder, mask_name)
        mask = Image.open(mask_path).convert("L")

        if self.transform:
            # Apply transformations
            image_gray = self.transform(image_gray)
            mask = self.transform(mask)

        return image_gray, mask

# Define paths for train, validation, and test sets
train_path = "/mnt/c/Users/kanek/University/Coding-Webe-Kuliah/Tugas/Semester-4/Deep_Learning/Project/Image_segmentation/train2" 
valid_path = "/mnt/c/Users/kanek/University/Coding-Webe-Kuliah/Tugas/Semester-4/Deep_Learning/Project/Image_segmentation/valid2"
test_path = "/mnt/c/Users/kanek/University/Coding-Webe-Kuliah/Tugas/Semester-4/Deep_Learning/Project/Image_segmentation/test2"

# Define transformations
image_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485], std=[0.229]),  # Assuming grayscale images
    transforms.Lambda(lambda x: x.clamp(0, 1))
])

# Create datasets
train_dataset = CustomDataset_general(train_path, transform=image_transform)
valid_dataset = CustomDataset_general(valid_path, transform=image_transform)
test_dataset = CustomDataset_general(test_path, transform=image_transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [3]:
print(len(train_loader))

376


In [4]:
if len(train_loader) > 0:
    x, y = next(iter(train_loader))
    print(x.shape , y.shape , type(x) , type(y))
else:
    print("Train loader is empty.")


torch.Size([4, 1, 224, 224]) torch.Size([4, 1, 224, 224]) <class 'torch.Tensor'> <class 'torch.Tensor'>


# Modelling

## Import Libraries

In [5]:
import torch
from torchvision import datasets
from torchmetrics.classification import JaccardIndex
from torch.utils.data import Dataset ,DataLoader
from torchvision.transforms import ToTensor
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.optim import lr_scheduler

## Loss

In [6]:
import torch.nn.functional as F

class DiceLoss(nn.Module):
    def __init__(self):
        super(DiceLoss, self).__init__()
        
    def forward(self, y_pred, y_true):
        y_pred = torch.sigmoid(y_pred)
        smooth = 1.0
        y_pred = y_pred.reshape(-1)
        y_true = y_true.reshape(-1)
        intersection = (y_pred * y_true).sum()
        
        dic_loss = 1 - (2.0 * intersection + smooth) / (y_pred.sum() + y_true.sum() + smooth)
        return dic_loss

## Import Backbone

In [7]:
import segmentation_models_pytorch as smp
import torchsummary


ENCODER = 'efficientnet-b0'
ENCODER_WEIGHTS = 'imagenet'
DEVICE = 'cuda'

ACTIVATION = None
model = smp.Unet(
    encoder_name=ENCODER, 
    encoder_weights=ENCODER_WEIGHTS, 
    in_channels=1, 
    classes=1, 
    activation=ACTIVATION,
)
model.to(DEVICE)
print(torchsummary.summary(model, (1, 224, 224)))

/home/wbchn/miniconda3/envs/PyTorch-DLN/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
         ZeroPad2d-1          [-1, 1, 225, 225]               0
Conv2dStaticSamePadding-2         [-1, 32, 112, 112]             288
       BatchNorm2d-3         [-1, 32, 112, 112]              64
MemoryEfficientSwish-4         [-1, 32, 112, 112]               0
         ZeroPad2d-5         [-1, 32, 114, 114]               0
Conv2dStaticSamePadding-6         [-1, 32, 112, 112]             288
       BatchNorm2d-7         [-1, 32, 112, 112]              64
MemoryEfficientSwish-8         [-1, 32, 112, 112]               0
          Identity-9             [-1, 32, 1, 1]               0
Conv2dStaticSamePadding-10              [-1, 8, 1, 1]             264
MemoryEfficientSwish-11              [-1, 8, 1, 1]               0
         Identity-12              [-1, 8, 1, 1]               0
Conv2dStaticSamePadding-13             [-1, 32, 1, 1]             288
         I

## Hyperparameters

In [11]:
checkpoints_path = "exp/checkpoints.pth"
os.makedirs("exp", exist_ok=True)

num_epochs = 10
lr=1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn = DiceLoss()

## Defining Model

In [12]:
from tqdm import tqdm
def train(model,optimizer,loss_fn,loader,device):
    epoch_loss =0.0
    model.train()
    
    for x,y in tqdm(loader):
        x = x.to(device, dtype = torch.float32)
        y = y.to(device, dtype = torch.float32)
        optimizer.zero_grad()
        pred = model(x)
        loss = loss_fn(pred,y)
        loss.backward()
        optimizer.step()
        epoch_loss +=loss.item()
        
    # Put Jaccard index here
    
    return epoch_loss/len(loader)
        

In [13]:
def valid(model,loader,loss_fn,device):
    epoch_loss = 0.0
    model.eval()
    
    for x,y in loader:
        x = x.to(device, dtype = torch.float32)
        y = y.to(device, dtype = torch.float32)
        pred = model(x)
        loss = loss_fn(pred,y)
        epoch_loss +=loss.item()
    return epoch_loss/len(loader)

## Model Training

In [14]:
from IPython.display import clear_output
from matplotlib import pyplot as plt

best_val_loss = float('inf')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_losses = []
num_no_improve = 0

model.to(device)  # Move the model to the appropriate device

for epoch in range(num_epochs):
    train_loss = train(model,optimizer,loss_fn,train_loader,device) # Train the model
    val_loss = valid(model,valid_loader,loss_fn,device) # Validate the model
    train_losses.append(train_loss) # Save the training loss
    
    clear_output(wait=True)
    print(f'Epoch {epoch+1}/{num_epochs}, Train Dice Loss: {train_loss:.4f}, Train Dice Similarity Coef: {1-train_loss:.4f}')
    print(f'Validation Dice Loss: {val_loss:.4f}, Validation Dice Similarity Coef: {1-val_loss:.4f}')
    
    if val_loss < best_val_loss: # Save the model with the best validation loss
        best_val_loss = val_loss
        print("Saving the model")
        torch.save(model.state_dict(), checkpoints_path)
    
    # Check for early stopping, If the validation loss does not improve for 5 consecutive epochs, stop the training
    if len(train_losses) > 1: 
        if val_loss <= best_val_loss: 
            num_no_improve += 1
        else:
            num_no_improve = 0
            
    if num_no_improve >= 5:
        print("Early stopping")
        break
        
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.title('Training Loss and Validation Loss')
plt.legend()
plt.show(block=False)

Epoch 2/10, Train Dice Loss: 0.4532, Train Dice Similarity Coef: 0.5468
Validation Dice Loss: 0.4965, Validation Dice Similarity Coef: 0.5035
Saving the model


 94%|█████████▎| 352/376 [05:28<00:22,  1.07it/s]


RuntimeError: CUDA error: unknown error
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


The Dice loss of 0.5882 and Dice Similiarity Coeefficient of 0.4118 on the train data indicated that more of the true image and predicted image on the train data is missclassified than the correctly classified, this shows us that the model is slightly underfitting. The model can be improved by increasing the number of epochs, increasing the learning rate, and increasing the batch size.

The Diceloss of the validation data is 0.4955, this shows that the model is neither underfitting nor overfitting. but not really performing well. The model can be improved by increasing the number of epochs, increasing the learning rate, and increasing the batch size.

The form of the predicted mask is also rectangular, which could be more suited to image detection tasks. The prediction maybe improved by using a different model approach.

## Model Inferencing

In [ ]:
def visualize_input_output_target(input_image, output_image, target_image):
    # Move tensors to CPU memory if they are on CUDA devices
    input_image = input_image.cpu()
    output_image = output_image.cpu()
    target_image = target_image.cpu()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot input image
    axes[0].imshow(input_image.squeeze().numpy(), cmap='gray')
    axes[0].set_title('Input Image')
    axes[0].axis('off')

    # Plot output image (predicted binary mask)
    axes[1].imshow(output_image.squeeze().numpy(), cmap='gray')
    axes[1].set_title('Output Image (Predicted)')
    axes[1].axis('off')

    # Plot target image (ground truth binary mask)
    axes[2].imshow(target_image.squeeze().numpy(), cmap='gray')
    axes[2].set_title('Target Image (Ground Truth)')
    axes[2].axis('off')

    plt.show()
    fig.savefig("output.png")

with torch.inference_mode():
    for batch, (X, y) in enumerate(valid_loader):
        X = X.to(DEVICE, dtype=torch.float32)
        y = y.to(DEVICE, dtype=torch.float32)

        y_pred_logits = model(X)
        
        # Assuming y_pred_logits contains probabilities or logits for each pixel
        # You may need to apply thresholding or other post-processing to get binary images
        # Here, we'll just consider values above 0.5 as foreground (1) and below as background (0)
        y_pred_binary = (y_pred_logits > 0.5).float()

        # Visualize input, output, and target binary images for the first batch
        visualize_input_output_target(X[0], y_pred_binary[0], y[0])